# Assignment 3, Person 2: ablation runs E5-E7, seed 2027 (Kaggle)

The 10 training ablations of the work plan, each changing **one** setting of the main system, all with
**seed 2027**. They are compared with the main run of the same seed
(`results/runs/20261124-084211_proposed_ft_syn_sv_seed2027`), which ran on Kaggle with the same GPU type and libraries
(seeds 1234 and 2027 were chosen for this reason, see DEVIATIONS.md D9).

The two T4 GPUs each work through their own queue of five runs, at the same time (about 3 hours in total):

| GPU | run name | ablation |
|---|---|---|
| GPU 0 | `abl_nospell` | E5 no spelling normalisation |
| GPU 0 | `abl_nostageb` | E7 no Stage B (Swedish fine-tuning) |
| GPU 0 | `abl_noera` | E7 no era-aware sampling |
| GPU 0 | `abl_hardlabels` | E7 hard pair labels |
| GPU 0 | `abl_layer4` | E6 layer 4 |
| GPU 1 | `abl_nostagea` | E7 no Stage A (synthetic English) |
| GPU 1 | `abl_nomarkers` | E7 no target markers |
| GPU 1 | `abl_layer8` | E6 layer 8 |
| GPU 1 | `abl_poolfirst` | E6 first-sub-word pooling |
| GPU 1 | `abl_poolsent` | E6 sentence-mean pooling |

Every run has its own `--name` (`proposed_abl_*`), so no ablation can replace the main run in the tables.

**Settings (right-hand panel > Session options):** Accelerator **GPU T4 x2**, Internet **On**.
Input: a dataset with `sentences.bin` (= `sentences.zip` renamed), e.g. Person 1's `senseshift-synthetic` (Add Input).
Run with **Save Version > Save & Run All (Commit)**: it runs in the background; the browser can be closed.
Everything is kept in `/kaggle/working` (the notebook's Output).

In [ ]:
# Cell 1: settings and the synthetic English set
SEED = 2027
PARALLEL = True          # True: both GPUs at once (~3 h). False: one GPU after the other (~6 h)
import os, glob, subprocess, threading, time, json, shutil
OUT  = '/kaggle/working'
RUNS = f'{OUT}/runs_a3_ablations'      # run folders (vectors/ included; the zip leaves them out)
LOGS = f'{OUT}/logs'
for d in (RUNS, LOGS): os.makedirs(d, exist_ok=True)

found = [p for p in glob.glob('/kaggle/input/**/generated_sentences', recursive=True) if os.path.isdir(p)]
if not found:
    archives = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
                if os.path.isfile(p) and os.path.basename(p).startswith('sentences')]
    assert archives, 'sentences.bin not found: Add Input > the dataset with sentences.bin'
    print('unpacking', archives[0], '(about 1 min)')
    !mkdir -p /tmp/kg/synthetic && unzip -q -o "{archives[0]}" -d /tmp/kg/synthetic
    found = [p for p in glob.glob('/tmp/kg/synthetic/**/generated_sentences', recursive=True) if os.path.isdir(p)]
GEN = found[0]
print('synthetic set:', GEN, '|', len(glob.glob(f'{GEN}/*/*.jsonl')), 'word files (2,347 expected)')

In [ ]:
# Cell 2: fresh git clone of the team repo, packages, GPUs
%cd /kaggle/working
!rm -rf /tmp/kg/SenseShift
!git clone -q https://github.com/abiha25/SenseShift-Semantic-change-across-multiple-time-periods.git /tmp/kg/SenseShift
%cd /tmp/kg/SenseShift
!git log --oneline -1
!pip install -q bcubed nltk pyyaml
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
n_gpu = torch.cuda.device_count()
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu >= 1, 'No GPU: set Accelerator to GPU T4 x2'
if n_gpu < 2 and PARALLEL:
    PARALLEL = False; print('Only one GPU: the runs go one after the other')

In [ ]:
# Cell 3: link the synthetic set; the clone must be clean (git status prints nothing)
!mkdir -p data/external/synthetic_en
!ln -sfn "{GEN}" data/external/synthetic_en/generated_sentences
!echo "word files via the link:" $(ls data/external/synthetic_en/generated_sentences/*/*.jsonl | wc -l)
!git status --porcelain

In [ ]:
# Cell 4: tests (expected: all passed, 0 failed)
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -1

## The runs
Each queue runs its five ablations one after the other on its own GPU. Each run writes its full log to `logs/<name>.log` and its GPU memory to `logs/<name>_gpu.csv`. A failed run does not stop the queue; it shows up as `FAILED` in Cell 6.

In [ ]:
# Cell 5: run both queues (about 3 h with PARALLEL = True)
QUEUES = {
 "0": [
  [
   "abl_nospell",
   [
    "preprocessing.spelling_normalisation=false"
   ]
  ],
  [
   "abl_nostageb",
   [
    "training.swedish_stage=false"
   ]
  ],
  [
   "abl_noera",
   [
    "training.era_aware_sampling=false"
   ]
  ],
  [
   "abl_hardlabels",
   [
    "training.soft_labels=false"
   ]
  ],
  [
   "abl_layer4",
   [
    "model.finetuned_layer=4"
   ]
  ]
 ],
 "1": [
  [
   "abl_nostagea",
   [
    "training.synthetic.enabled=false"
   ]
  ],
  [
   "abl_nomarkers",
   [
    "model.target_markers=false"
   ]
  ],
  [
   "abl_layer8",
   [
    "model.finetuned_layer=8"
   ]
  ],
  [
   "abl_poolfirst",
   [
    "model.pooling=first_subword"
   ]
  ],
  [
   "abl_poolsent",
   [
    "model.pooling=sentence_mean"
   ]
  ]
 ]
}
QUEUES = {int(k): v for k, v in QUEUES.items()}

def run_queue(gpu, runs, gpu_id_for_env):
    ckpt = f'/tmp/kg/checkpoints_gpu{gpu}'      # Stage A checkpoints, one folder per queue
    os.makedirs(ckpt, exist_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu_id_for_env))
    for name, overrides in runs:
        cmd = ['python', '-m', 'src.proposed', 'run', '--name', name, '--set', f'seed={SEED}',
               '--set', f'output.results_dir={RUNS}', '--set', f'output.checkpoints_dir={ckpt}']
        for o in overrides:
            cmd += ['--set', o]
        t0 = time.time()
        smi = subprocess.Popen(['nvidia-smi', f'--id={gpu_id_for_env}', '--query-gpu=timestamp,memory.used',
                                '--format=csv,noheader,nounits', '-l', '5'],
                               stdout=open(f'{LOGS}/{name}_gpu.csv', 'w'), stderr=subprocess.DEVNULL)
        with open(f'{LOGS}/{name}.log', 'w') as log:
            rc = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
        smi.terminate()
        print(f'[GPU {gpu}] {name}: exit {rc}, {(time.time() - t0) / 60:.1f} min', flush=True)
        # Stage A of abl_nospell is the same model as the main run's Stage A (spelling is not
        # part of the Stage A key), and abl_nostageb reuses it. Delete checkpoints after that.
        if name != 'abl_nospell':
            shutil.rmtree(ckpt, ignore_errors=True); os.makedirs(ckpt, exist_ok=True)

if PARALLEL:
    threads = [threading.Thread(target=run_queue, args=(g, runs, g)) for g, runs in QUEUES.items()]
    for t in threads: t.start()
    for t in threads: t.join()
else:
    for g, runs in QUEUES.items():
        run_queue(g, runs, 0)
print('all runs finished')

## Check the runs
Every run must have `scores.json`, the same commit, `uncommitted = False`, and a `proposed_abl_*` name. Cell 7 checks that the Stage A of `abl_nospell` (same Stage A key as the main run) gives the same held-out AUC as the main run of this seed: if it does, Kaggle reproduced the main run's Stage A exactly, so the ablation differences are not platform effects.

In [ ]:
# Cell 6: summary of the ablation runs
import pandas as pd
names = [n for runs in QUEUES.values() for n, _ in runs]
rows = []
for name in names:
    d = sorted(glob.glob(f'{RUNS}/*_proposed_{name}_seed{SEED}'))
    if not d or not os.path.exists(f'{d[-1]}/scores.json'):
        rows.append({'run': name, 'status': 'FAILED (see logs/' + name + '.log)'}); continue
    d = d[-1]
    info = json.load(open(f'{d}/run_info.json')); s = json.load(open(f'{d}/scores.json'))
    sel = json.load(open(f'{d}/selection.json')); git = info.get('git') or {}
    sh = s.get('subtask1', {}).get('subtask1b', {}).get('shuffle_control') or {}
    mem = [int(l.split(',')[1]) for l in open(f'{LOGS}/{name}_gpu.csv') if ',' in l]
    rows.append({'run': os.path.basename(d), 'status': 'ok', 'commit': (git.get('commit') or 'NONE')[:7],
                 'uncommitted': git.get('uncommitted_changes'), 'torch': info['environment']['packages'].get('torch'),
                 '1a': s['scores'].get('1a_bcubed_f1'), '1b': s['scores'].get('1b_spearman'),
                 '1b shuffle': sh.get('shuffled_mean'), '2': s['scores'].get('2_macro_f1'),
                 'APD': s['scores'].get('graded_spearman'),
                 'minutes': round(sel.get('timing', {}).get('total_seconds', float('nan')) / 60, 1),
                 'peak GPU MiB': max(mem) if mem else None})
table = pd.DataFrame(rows).round(3)
table.to_csv(f'{OUT}/a3_ablations_seed{SEED}_summary.csv', index=False)
assert not any(f'_proposed_ft_syn_sv_seed' in p for p in os.listdir(RUNS)), 'a run without --name exists!'
table

In [ ]:
# Cell 7: does Kaggle reproduce the main run's Stage A? (platform check for D9)
main = 'results/runs/20261124-084211_proposed_ft_syn_sv_seed2027/training_history.json'
mine = sorted(glob.glob(f'{RUNS}/*_proposed_abl_nospell_seed{SEED}/training_history.json'))
if mine:
    a = [e['val'] for e in json.load(open(main))['stage_a']['epochs']]
    b = [e['val'] for e in json.load(open(mine[-1]))['stage_a']['epochs']]
    print('main run   Stage A held-out AUC:', [round(x, 4) for x in a])
    print('abl_nospell Stage A held-out AUC:', [round(x, 4) for x in b])
    print('IDENTICAL' if a == b else f'DIFFERENT (max difference {max(abs(x - y) for x, y in zip(a, b)):.2e})')
else:
    print('abl_nospell did not finish')

In [ ]:
# Cell 8: one small zip without the vectors (for the repo); full folders stay in the Output
!cd /kaggle/working && rm -f a3_ablations_seed2027.zip && zip -qr a3_ablations_seed2027.zip runs_a3_ablations logs a3_ablations_seed2027_summary.csv -x "*/vectors/*"
!ls -la /kaggle/working